# Market Intelligence Exploratory Data Analysis (EDA)
**Student:** Mohamed Abd Elsalam Mohamed Gharieb  
**Program:** AWS AI & ML Scholars Program — Future AWS Agentic AI Business Professional  
**Dataset:** Bitext — Training Dataset for Chatbots / Virtual Assistants (Kaggle, CDLA-Sharing-1.0)  
**Analytical Guardrail:** This dataset contains synthetic/curated English support utterances. It informs support intent taxonomy, linguistic robustness requirements, and AI workflow design, **not** MENA market size or SME willingness-to-pay.


In [1]:
import pandas as pd
import numpy as np
from collections import Counter

df = pd.read_csv('../03_DATASET/raw/20000-Utterances-Training-dataset-for-chatbots-virtual-assistant-Bitext-sample.csv')
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Columns:", list(df.columns))
print("\nMissing values per column:\n", df.isnull().sum().to_dict())
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate utterance texts: {df['utterance'].duplicated().sum()}")
print(f"Unique categories: {df['category'].nunique()} | Unique intents: {df['intent'].nunique()}")


Shape: 21,534 rows x 6 columns
Columns: ['flags', 'utterance', 'category', 'intent', 'char_len', 'word_len']

Missing values per column:
{'flags': 0, 'utterance': 0, 'category': 0, 'intent': 0, 'char_len': 0, 'word_len': 0}
Exact duplicate rows: 12
Duplicate utterance texts: 21
Unique categories: 11 | Unique intents: 27


In [2]:
cat_summary = df['category'].value_counts().rename_axis('category').reset_index(name='count')
cat_summary['share_pct'] = (cat_summary['count'] / len(df) * 100).round(2)
print(cat_summary.to_string(index=False))


        category  count  share_pct
         PAYMENT   4636      21.53
         ACCOUNT   4557      21.16
         CONTACT   3081      14.31
        INVOICES   2443      11.34
           ORDER   2256      10.48
         REFUNDS   1932       8.97
        FEEDBACK   1326       6.16
        DELIVERY    501       2.33
CANCELLATION_FEE    360       1.67
      NEWSLETTER    236       1.10
        SHIPPING    206       0.96


In [3]:
intent_summary = df.groupby(['category', 'intent']).size().reset_index(name='count').sort_values('count', ascending=False)
intent_summary['share_pct'] = (intent_summary['count'] / len(df) * 100).round(2)
print(intent_summary.to_string(index=False))


        category                   intent  count  share_pct
         PAYMENT            payment_issue   4366      20.27
         ACCOUNT           create_account   2122       9.85
         CONTACT contact_customer_service   2055       9.54
        INVOICES              get_invoice   1430       6.64
           ORDER              track_order   1224       5.68
         REFUNDS               get_refund   1150       5.34
         CONTACT      contact_human_agent   1026       4.76
        INVOICES           check_invoices   1013       4.70
         ACCOUNT         recover_password    986       4.58
           ORDER             change_order    926       4.30
         ACCOUNT           delete_account    913       4.24
        FEEDBACK                complaint    746       3.46
        FEEDBACK                   review    580       2.69
         REFUNDS      check_refund_policy    479       2.22
        DELIVERY         delivery_options    360       1.67
CANCELLATION_FEE   check_cancellation_fe

In [4]:
df['char_len'] = df['utterance'].astype(str).str.len()
df['word_len'] = df['utterance'].astype(str).str.split().str.len()
print("Text Length Summary:")
print(df[['char_len', 'word_len']].describe().round(2).to_string())

flag_counts = Counter(ch for fl in df['flags'].astype(str) for ch in fl)
for k, v in flag_counts.most_common():
    print(f"Flag {k}: {v:,} rows ({v/len(df)*100:.2f}%)")


Text Length Summary:
       char_len  word_len
count  21534.00  21534.00
mean      52.54     10.86
std       16.79      3.72
min       10.00      2.00
25%       41.00      8.00
50%       51.00     10.00
75%       61.00     13.00
max      128.00     28.00

Flag B: 21,534 rows (100.00%)
Flag L: 18,752 rows (87.08%)
Flag I: 12,766 rows (59.28%)
Flag C: 10,911 rows (50.67%)
Flag Q: 9,552 rows (44.36%)
Flag Z: 3,412 rows (15.84%)
Flag P: 2,888 rows (13.41%)
Flag M: 2,836 rows (13.17%)
Flag D: 2,596 rows (12.06%)
Flag S: 1,884 rows (8.75%)
Flag W: 1,047 rows (4.86%)
Flag E: 392 rows (1.82%)


In [5]:
ana = pd.read_csv('./analysis_results.csv')
tier_rollup = ana.groupby('automation_tier')['count'].sum().reset_index()
tier_rollup['share_pct'] = (tier_rollup['count'] / len(df) * 100).round(2)
print(tier_rollup.sort_values('count', ascending=False).to_string(index=False))


                    automation_tier  count  share_pct
Tier 4 - Human-in-the-Loop Priority  10292      47.79
         Tier 2 - API Status Lookup   4956      23.01
      Tier 3 - Authenticated Action   4546      21.11
        Tier 1 - Knowledge-Base RAG   1740       8.08
